In [0]:
# ==============================
# STEP 1: Imports
# ==============================
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ==============================
# STEP 2: Read Bronze table
# ==============================
df_raw = spark.table("udp.bronze.counterparties")

display(df_raw)

In [0]:
# ==============================
# STEP 3: Trim & standardize
# ==============================
df_clean = (
    df_raw
    .withColumn("name", trim(col("name")))
    .withColumn("country", upper(col("country")))
)

# ==============================
# STEP 4: Parse onboarded_date
# ==============================
df_clean = df_clean.withColumn(
    "onboarded_date_parsed",
    coalesce(
        try_to_date("onboarded_date", "yyyy-MM-dd"),
        try_to_date("onboarded_date", "dd/MM/yyyy")
    )
)

In [0]:
# ==============================
# STEP 5: Bad records
# ==============================
df_bad = df_clean.filter(
    col("name").isNull() |
    col("country").isNull() |
    col("onboarded_date_parsed").isNull() |
    (col("onboarded_date_parsed") > current_date())
)

# ==============================
# STEP 6: Good records
# ==============================
df_good = df_clean.subtract(df_bad)

display(df_bad)

In [0]:
# ==============================
# STEP 7: Credit rating check
# ==============================
valid_ratings = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]

df_invalid_rating = df_good.filter(
    ~col("credit_rating").isin(valid_ratings)
)

df_good = df_good.filter(
    col("credit_rating").isin(valid_ratings)
)

# Add invalid ratings to bad
df_bad = df_bad.union(df_invalid_rating)

In [0]:
# ==============================
# STEP 8: Deduplicate
# ==============================
window_spec = Window.partitionBy("counterparty_id") \
                    .orderBy(col("onboarded_date_parsed").desc())

df_final = (
    df_good
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

display(df_final)

In [0]:
# ==============================
# STEP 9: Write SILVER table
# ==============================
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("udp.silver.counterparties")

